# Country sensitivity and leave-one-country-out analysis

**REDLINC XII | Computational workflow | R**

Assess the stability of model estimates when countries are omitted sequentially.

> **Code-only notebook.** This file contains no embedded execution outputs. It requires locally configured inputs and has not been validated end-to-end in a clean environment.


## Before running

1. Install the packages described in the repository README.
2. Configure local input and output locations without committing any local files.
3. Review variable coding, model assumptions and execution order.
4. Keep generated outputs outside version control.


## Analysis block 1

The following code block is preserved from the reviewed draft. Read its internal comments for detailed operations and verify all required inputs before execution.


In [ ]:
# =============================================================================
# SEM V7 — LOCO SENSITIVITY ANALYSIS (Leave-One-Country-Out)
# Purpose: Verify that no single country drives the primary BMI->cognition
#          results. Runs SEM V7 syntax 8 times (7 LOCO + 1 full sample),
#          using MLR + FIML (no bootstrap) for speed.
# Output:  LOCO_Coeficientes_Diretos.csv
#          LOCO_Efeitos_Indirectos.csv
#          LOCO_Fit_Indices.csv
#          LOCO_Resumo_Sensibilidade.csv
# Requires: locally supplied analysis_input.xlsx (not distributed)
#           Identical variable construction to SEM V7 Cell 1
# =============================================================================

library(openxlsx)
library(dplyr)
library(lavaan)

out_dir <- Sys.getenv("SEM_WORKDIR", unset = "./")

# =============================================================================
# 1. LOAD AND PREPARE DATA (identical to V7 Cell 1)
# =============================================================================
cat("Loading data...\n")
dat <- read.xlsx(paste0(out_dir, "analysis_input.xlsx"))
cat(sprintf("  Raw: %d rows, %d columns\n", nrow(dat), ncol(dat)))

# Identify and immediately rename country column to avoid encoding issues
pais_col <- "Pa&#237;s"
cat(sprintf("  Country column detected: '%s' -> renaming to 'Country'\n", pais_col))
names(dat)[names(dat) == pais_col] <- "Country"

cat("  N per country (raw):\n")
print(sort(table(dat[["Country"]]), decreasing = TRUE))

# Derived variables (identical to V7)
dat$Ooforectomia <- as.integer(!is.na(dat$EdadOoforecto))
dat <- dat %>% mutate(GynSurgery = Histerectom + Ooforectomia)

dat <- dat %>%
  mutate(
    MRS_Somatic_Limpia = MRS1 + MRS2 + MRS4,
    MRS_Psych_Total    = MRS5 + MRS6 + MRS7 + MRS8,
    MRS_Urogenital_Tot = MRS9 + MRS10 + MRS11
  )

sarcf_items <- c("SARCF","SARCF_A","SARCF_B","SARCF_C","SARCF_D")
dat[sarcf_items] <- lapply(dat[sarcf_items],
                           function(x) as.numeric(as.character(x)))
dat$SARCF_Total <- rowSums(dat[, sarcf_items], na.rm = FALSE)

dat$MenoYears <- dat$Edad - dat$MenoDesde

# BMI groups — exclude Underweight
# Country already renamed above, safe to use directly
dat <- dat %>%
  filter(!is.na(IMC), IMC >= 18.5) %>%
  mutate(
    IMC_Overweight  = as.numeric(IMC >= 25 & IMC < 30),
    IMC_Obese       = as.numeric(IMC >= 30),
    IMC_Group_Label = case_when(
      IMC >= 18.5 & IMC < 25 ~ "Normal",
      IMC >= 25   & IMC < 30 ~ "Overweight",
      IMC >= 30               ~ "Obese"
    ),
    Country = as.character(Country)
  )

# MoCA education correction
dat$MoCA_corrected <- dat$MOCATOTAL2
dat$MoCA_corrected[!is.na(dat$Estudio) & dat$Estudio <= 12] <-
  dat$MoCA_corrected[!is.na(dat$Estudio) & dat$Estudio <= 12] + 1
dat$MOCA_OUT <- dat$MoCA_corrected

# Complete cases on model variables
model_vars <- c("Jenkins1","Jenkins2","Jenkins3","Jenkins4",
                "MRS_Somatic_Limpia","MRS_Psych_Total","MRS_Urogenital_Tot",
                "IMC_Overweight","IMC_Obese",
                "GynSurgery","Edad","MenoYears","THMTotal",
                "HipertArt","Diabete","Colesterol",
                "ActFisica","SARCF_Total","MOCA_OUT","Estudio","Country")

df_analysis <- dat[complete.cases(dat[, model_vars]), model_vars]
numeric_vars <- setdiff(model_vars, "Country")
df_analysis[numeric_vars] <- lapply(df_analysis[numeric_vars],
                                    function(x) as.numeric(as.character(x)))

cat(sprintf("\ndf_analysis: N=%d\n", nrow(df_analysis)))
cat("N per country in analysis sample:\n")
country_n <- sort(table(df_analysis$Country), decreasing = TRUE)
print(country_n)
countries <- names(country_n)

# =============================================================================
# 2. MODEL SYNTAX V7 (MLR version — no bootstrap, uses := for indirects)
# =============================================================================
syntax_v7_mlr <- '
  Jenkins_Latent =~ Jenkins1 + Jenkins2 + Jenkins3 + Jenkins4
  MRS_Latent     =~ MRS_Somatic_Limpia + MRS_Psych_Total + MRS_Urogenital_Tot

  Jenkins_Latent ~ a_sob*IMC_Overweight + a_obe*IMC_Obese +
                   a_gyn*GynSurgery + a_age*Edad + a_meno*MenoYears +
                   a_thm*THMTotal + a_hyp*HipertArt +
                   a_dia*Diabete + a_dislip*Colesterol

  MRS_Latent     ~ b_sob*IMC_Overweight + b_obe*IMC_Obese +
                   b_slp*Jenkins_Latent + b_gyn*GynSurgery +
                   b_thm*THMTotal + b_act*ActFisica + b_meno*MenoYears +
                   b_hyp*HipertArt + b_dia*Diabete + b_dislip*Colesterol

  SARCF_Total    ~ c_sob*IMC_Overweight + c_obe*IMC_Obese +
                   c_mrs*MRS_Latent + c_gyn*GynSurgery +
                   c_age*Edad + c_act*ActFisica + c_meno*MenoYears +
                   c_hyp*HipertArt + c_dia*Diabete + c_dislip*Colesterol

  MOCA_OUT       ~ d_sob*IMC_Overweight + d_obe*IMC_Obese +
                   d_mrs*MRS_Latent + d_srf*SARCF_Total +
                   d_gyn*GynSurgery + d_edu*Estudio + d_age*Edad +
                   d_meno*MenoYears + d_hyp*HipertArt +
                   d_dia*Diabete + d_dislip*Colesterol

  # Indirect effects
  sob_Sleep_MRS_MoCA       := a_sob * b_slp * d_mrs
  sob_Sleep_MRS_SARCF_MoCA := a_sob * b_slp * c_mrs * d_srf
  sob_MRS_MoCA             := b_sob * d_mrs
  sob_MRS_SARCF_MoCA       := b_sob * c_mrs * d_srf
  sob_SARCF_MoCA           := c_sob * d_srf
  sob_Total_Indirect       := (a_sob*b_slp*d_mrs) + (a_sob*b_slp*c_mrs*d_srf) +
                              (b_sob*d_mrs) + (b_sob*c_mrs*d_srf) + (c_sob*d_srf)

  obe_Sleep_MRS_MoCA       := a_obe * b_slp * d_mrs
  obe_Sleep_MRS_SARCF_MoCA := a_obe * b_slp * c_mrs * d_srf
  obe_MRS_MoCA             := b_obe * d_mrs
  obe_MRS_SARCF_MoCA       := b_obe * c_mrs * d_srf
  obe_SARCF_MoCA           := c_obe * d_srf
  obe_Total_Indirect       := (a_obe*b_slp*d_mrs) + (a_obe*b_slp*c_mrs*d_srf) +
                              (b_obe*d_mrs) + (b_obe*c_mrs*d_srf) + (c_obe*d_srf)
'

# =============================================================================
# 3. FIT FUNCTION
# =============================================================================
fit_sem_mlr <- function(data, label) {
  fit <- tryCatch(
    sem(syntax_v7_mlr, data = data,
        estimator = "ML", fixed.x = FALSE),
    error = function(e) {
      cat(sprintf("  ERROR [%s]: %s\n", label, conditionMessage(e)))
      return(NULL)
    }
  )
  if (is.null(fit)) return(NULL)
  if (!lavInspect(fit, "converged")) {
    cat(sprintf("  WARNING [%s]: did not converge.\n", label))
    return(NULL)
  }
  return(fit)
}

# =============================================================================
# 4. FULL MODEL (reference)
# =============================================================================
cat("\n--- Fitting FULL MODEL (reference) ---\n")
fit_full <- fit_sem_mlr(df_analysis, "Full")
if (is.null(fit_full)) stop("Full model failed — check data and syntax.")

fi_full <- fitMeasures(fit_full,
                       c("cfi","tli","rmsea","srmr"))
cat(sprintf("  Full model fit: CFI=%.3f TLI=%.3f RMSEA=%.3f SRMR=%.3f\n",
            fi_full["cfi"], fi_full["tli"],
            fi_full["rmsea"], fi_full["srmr"]))

# =============================================================================
# 5. LOCO LOOP
# =============================================================================
cat(sprintf("\nRunning LOCO for %d countries...\n", length(countries)))

direct_results   <- list()
indirect_results <- list()
fit_results      <- list()

# Helper: extract direct BMI effects
extract_direct <- function(fit, label, n) {
  pe <- parameterEstimates(fit, standardized = TRUE)
  bmi_rows <- pe[pe$op == "~" & pe$rhs %in% c("IMC_Overweight","IMC_Obese"), ]
  bmi_rows$Country_Left_Out <- label
  bmi_rows$N_used           <- n
  bmi_rows[, c("Country_Left_Out","N_used",
               "lhs","rhs","est","se","z","pvalue","std.all")]
}

# Helper: extract indirect effects
extract_indirect <- function(fit, label, n) {
  pe  <- parameterEstimates(fit)
  ind <- pe[pe$op == ":=", ]
  ind$Country_Left_Out <- label
  ind$N_used           <- n
  ind[, c("Country_Left_Out","N_used","lhs","est","se","pvalue")]
}

# Full model as reference
N_full <- nrow(df_analysis)
direct_results[["Full"]]   <- extract_direct(fit_full,  sprintf("Full (N=%d)", N_full), N_full)
indirect_results[["Full"]] <- extract_indirect(fit_full, sprintf("Full (N=%d)", N_full), N_full)
fit_results[["Full"]] <- data.frame(
  Country_Left_Out = sprintf("Full (N=%d)", N_full),
  N_used    = N_full,
  CFI       = round(fi_full["cfi"],  3),
  TLI       = round(fi_full["tli"],  3),
  RMSEA     = round(fi_full["rmsea"],3),
  SRMR      = round(fi_full["srmr"],        3),
  Converged = TRUE
)

# LOCO iterations
for (ctry in countries) {
  df_loco <- df_analysis[df_analysis$Country != ctry, ]
  n_loco  <- nrow(df_loco)
  label   <- sprintf("LOCO: -%s (N=%d)", ctry, n_loco)
  cat(sprintf("  Fitting: leave out %-12s | N_remaining=%d\n", ctry, n_loco))

  fit_loco <- fit_sem_mlr(df_loco, label)

  if (!is.null(fit_loco)) {
    fi <- fitMeasures(fit_loco,
                      c("cfi","tli","rmsea","srmr"))
    cat(sprintf("    CFI=%.3f TLI=%.3f RMSEA=%.3f SRMR=%.3f\n",
                fi["cfi"], fi["tli"],
                fi["rmsea"], fi["srmr"]))

    direct_results[[ctry]]   <- extract_direct(fit_loco,  label, n_loco)
    indirect_results[[ctry]] <- extract_indirect(fit_loco, label, n_loco)

    fit_results[[ctry]] <- data.frame(
      Country_Left_Out = label,
      N_used    = n_loco,
      CFI       = round(fi["cfi"],  3),
      TLI       = round(fi["tli"],  3),
      RMSEA     = round(fi["rmsea"],3),
      SRMR      = round(fi["srmr"],        3),
      Converged = TRUE
    )
  } else {
    fit_results[[ctry]] <- data.frame(
      Country_Left_Out = label, N_used = n_loco,
      CFI = NA, TLI = NA, RMSEA = NA, SRMR = NA,
      Converged = FALSE
    )
  }
}

# =============================================================================
# 6. COMBINE AND EXPORT
# =============================================================================
cat("\nCompiling results...\n")

direct_all <- do.call(rbind, direct_results)
rownames(direct_all) <- NULL
write.csv(direct_all,
          paste0(out_dir, "LOCO_Coeficientes_Diretos.csv"),
          row.names = FALSE)
cat("OK LOCO_Coeficientes_Diretos.csv saved.\n")

indirect_all <- do.call(rbind, indirect_results)
rownames(indirect_all) <- NULL
write.csv(indirect_all,
          paste0(out_dir, "LOCO_Efeitos_Indirectos.csv"),
          row.names = FALSE)
cat("OK LOCO_Efeitos_Indirectos.csv saved.\n")

fit_all <- do.call(rbind, fit_results)
rownames(fit_all) <- NULL
write.csv(fit_all,
          paste0(out_dir, "LOCO_Fit_Indices.csv"),
          row.names = FALSE)
cat("OK LOCO_Fit_Indices.csv saved.\n")

# =============================================================================
# 7. SENSITIVITY SUMMARY
# =============================================================================
cat("\nComputing sensitivity summary...\n")

full_direct <- direct_results[["Full"]]
sensitivity_rows <- list()

for (ctry in countries) {
  if (!ctry %in% names(direct_results)) next
  loco_direct <- direct_results[[ctry]]

  for (i in 1:nrow(full_direct)) {
    lhs_i  <- full_direct$lhs[i]
    rhs_i  <- full_direct$rhs[i]
    b_full <- full_direct$std.all[i]
    p_full <- full_direct$pvalue[i]

    loco_row <- loco_direct[loco_direct$lhs == lhs_i &
                              loco_direct$rhs == rhs_i, ]
    if (nrow(loco_row) == 0) next

    b_loco <- loco_row$std.all[1]
    p_loco <- loco_row$pvalue[1]

    rel_change <- ifelse(abs(b_full) > 0.001,
                         (b_loco - b_full) / abs(b_full) * 100, NA)

    sig_flip <- (p_full < 0.05) != (p_loco < 0.05)

    sensitivity_rows[[length(sensitivity_rows)+1]] <- data.frame(
      Country_Left_Out = ctry,
      N_left_out       = as.integer(country_n[ctry]),
      Dependent        = lhs_i,
      Predictor        = rhs_i,
      Beta_Full        = round(b_full, 4),
      Beta_LOCO        = round(b_loco, 4),
      Rel_Change_pct   = round(rel_change, 1),
      p_Full           = round(p_full, 4),
      p_LOCO           = round(p_loco, 4),
      Sig_Flip         = sig_flip,
      stringsAsFactors = FALSE
    )
  }
}

sensitivity <- do.call(rbind, sensitivity_rows)
rownames(sensitivity) <- NULL
sensitivity$Influential <- abs(sensitivity$Rel_Change_pct) > 10 | sensitivity$Sig_Flip

write.csv(sensitivity,
          paste0(out_dir, "LOCO_Resumo_Sensibilidade.csv"),
          row.names = FALSE)
cat("OK LOCO_Resumo_Sensibilidade.csv saved.\n")

# =============================================================================
# 8. CONSOLE SUMMARY
# =============================================================================
cat("\n", paste(rep("=", 65), collapse=""), "\n")
cat("LOCO SENSITIVITY SUMMARY\n")
cat(paste(rep("=", 65), collapse=""), "\n")

cat("\nModel fit across LOCO iterations:\n")
print(fit_all[, c("Country_Left_Out","N_used","CFI","TLI","RMSEA","SRMR")],
      row.names = FALSE)

cat("\nInfluential countries (>10% beta change OR significance flip):\n")
inf_rows <- sensitivity[sensitivity$Influential == TRUE, ]
if (nrow(inf_rows) > 0) {
  print(inf_rows[order(-abs(inf_rows$Rel_Change_pct)),
                 c("Country_Left_Out","Dependent","Predictor",
                   "Beta_Full","Beta_LOCO","Rel_Change_pct",
                   "p_Full","p_LOCO","Sig_Flip")],
        row.names = FALSE)
} else {
  cat("  None — results are stable across all LOCO iterations.\n")
}

cat("\nMaximum relative beta change per coefficient (any country):\n")
max_change <- sensitivity %>%
  group_by(Dependent, Predictor) %>%
  summarise(
    Max_Rel_Change_pct    = round(max(abs(Rel_Change_pct), na.rm=TRUE), 1),
    Country_Max_Influence = Country_Left_Out[which.max(abs(Rel_Change_pct))],
    Any_Sig_Flip          = any(Sig_Flip),
    .groups = "drop"
  ) %>%
  arrange(desc(Max_Rel_Change_pct))
print(as.data.frame(max_change), row.names=FALSE)

cat("\nOutput files saved to", out_dir, ":\n")
for (f in c("LOCO_Coeficientes_Diretos.csv",
            "LOCO_Efeitos_Indirectos.csv",
            "LOCO_Fit_Indices.csv",
            "LOCO_Resumo_Sensibilidade.csv"))
  cat(sprintf("  %s\n", f))
cat(paste(rep("=", 65), collapse=""), "\n")
cat("LOCO analysis complete.\n")

## Analysis block 2

The following code block is preserved from the reviewed draft. Read its internal comments for detailed operations and verify all required inputs before execution.


## Reproducibility and interpretation

Check execution logs, convergence and assumptions before interpreting any estimates. This repository intentionally excludes study-specific outputs. For scientific interpretation, refer to the associated publication when available.
